In [14]:
from core.db import get_engine
from core.db_objects import _get_columns
from core.libs import pd, re
from collections import defaultdict

engine = get_engine()


💻 Conectado a la base de datos helloworldtree


In [15]:
DISCOVERY_SQL = """
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema = 'public'
  AND table_type = 'BASE TABLE'
  AND table_name LIKE 'inventory\\_%' ESCAPE '\\'
ORDER BY table_name;
"""

from sqlalchemy import text

with engine.connect() as conn:
    inventory_tables = pd.read_sql(text(DISCOVERY_SQL), conn)

inventory_tables.head(16)



,table_schema,table_name
0,public,inventory_cr_2021
1,public,inventory_cr_2022
2,public,inventory_cr_2024
3,public,inventory_cr_2025
4,public,inventory_gt_2021
5,public,inventory_gt_2022
6,public,inventory_gt_2024
7,public,inventory_gt_2025
8,public,inventory_mx_2021
9,public,inventory_mx_2022


In [16]:
REGIONS = {"cr", "gt", "mx", "us"}
pattern = re.compile(r"^inventory_([a-z]{2})_(\d{4})$")

records = []

for _, row in inventory_tables.iterrows():
    name = row["table_name"]
    match = pattern.match(name)
    if not match:
        continue

    region, year = match.group(1), int(match.group(2))
    if region not in REGIONS:
        continue

    records.append({
        "schema": row["table_schema"],
        "table_name": name,
        "region": region,
        "year": year
    })

inventories = (
    pd.DataFrame(records)
    .sort_values(["region", "year"])
    .reset_index(drop=True)
)

inventories


,schema,table_name,region,year
0,public,inventory_cr_2021,cr,2021
1,public,inventory_cr_2022,cr,2022
2,public,inventory_cr_2024,cr,2024
3,public,inventory_cr_2025,cr,2025
4,public,inventory_gt_2021,gt,2021
5,public,inventory_gt_2022,gt,2022
6,public,inventory_gt_2024,gt,2024
7,public,inventory_gt_2025,gt,2025
8,public,inventory_mx_2021,mx,2021
9,public,inventory_mx_2022,mx,2022


In [17]:
schemas = {}

for _, row in inventories.iterrows():
    table = row["table_name"]
    schema = row["schema"]

    cols = _get_columns(engine, schema, table)

    schemas[table] = (
        pd.DataFrame([
            {
                "column_name": col,
                "data_type": meta[0],
                "udt_name": meta[1]
            }
            for col, meta in cols.items()
        ])
        .sort_values("column_name")
        .reset_index(drop=True)
    )

len(schemas)


16

In [18]:
for table, df in schemas.items():
    print(f"\n=== {table} ===")
    display(df)



=== inventory_cr_2021 ===


,column_name,data_type,udt_name
0,alive_tree,double precision,float8
1,contractcode,text,text
2,coppiced_id,text,text
3,cruise_date,date,date
4,cruisedate,date,date
5,dbh_in,double precision,float8
6,dead_tree,double precision,float8
7,defect_ht_ft,double precision,float8
8,defect_id,text,text
9,disease_id,text,text



=== inventory_cr_2022 ===


,column_name,data_type,udt_name
0,alive_tree,double precision,float8
1,contractcode,text,text
2,coppiced_id,text,text
3,cruise_date,date,date
4,cruisedate,date,date
5,dbh_in,numeric,numeric
6,dead_tree,double precision,float8
7,defect_ht_ft,numeric,numeric
8,defect_id,text,text
9,disease_id,text,text



=== inventory_cr_2024 ===


...

,column_name,data_type,udt_name
0,alive_tree,double precision,float8
1,contractcode,text,text
2,coppiced_id,text,text
3,cruise_date,date,date
4,cruisedate,date,date
5,dbh_in,numeric,numeric
6,dead_tree,double precision,float8
7,defect_ht_ft,numeric,numeric
8,defect_id,text,text
9,disease_id,text,text



=== inventory_us_2024 ===


,column_name,data_type,udt_name
0,alive_tree,double precision,float8
1,contractcode,text,text
2,coppiced_id,text,text
3,cruisedate,timestamp without time zone,timestamp
4,dbh_in,numeric,numeric
5,dead_tree,double precision,float8
6,defect_ht_ft,numeric,numeric
7,defect_id,text,text
8,disease_id,text,text
9,doyle_bf,numeric,numeric



=== inventory_us_2025 ===


,column_name,data_type,udt_name
0,alive_tree,double precision,float8
1,contractcode,text,text
2,coppiced_id,text,text
3,cruisedate,timestamp without time zone,timestamp
4,dbh_in,numeric,numeric
5,dead_tree,double precision,float8
6,defect_ht_ft,numeric,numeric
7,defect_id,text,text
8,disease_id,text,text
9,doyle_bf,numeric,numeric


In [19]:
from core.libs import pd

# 1. Superset de columnas (schema más amplio)
all_columns = sorted(
    {col for df in schemas.values() for col in df["column_name"].tolist()}
)

# 2. Base del dataframe
matrix = pd.DataFrame({"canonical_column": all_columns})

# 3. Rellenar por inventory
for table, df in schemas.items():
    col_types = dict(zip(df["column_name"], df["data_type"]))
    matrix[table] = matrix["canonical_column"].map(
        lambda c: col_types.get(c, "")
    )

matrix


,canonical_column,inventory_cr_2021,inventory_cr_2022,inventory_cr_2024,inventory_cr_2025,inventory_gt_2021,inventory_gt_2022,inventory_gt_2024,inventory_gt_2025,inventory_mx_2021,inventory_mx_2022,inventory_mx_2024,inventory_mx_2025,inventory_us_2021,inventory_us_2022,inventory_us_2024,inventory_us_2025
0,alive_tree,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision
1,contractcode,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text
2,coppiced_id,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text
3,cruise_date,date,date,,,date,date,,,date,date,,,date,date,,
4,cruisedate,date,date,timestamp without time zone,timestamp without time zone,date,date,timestamp without time zone,timestamp without time zone,date,date,timestamp without time zone,timestamp without time zone,date,date,timestamp without time zone,timestamp without time zone
5,dbh_in,double precision,numeric,double precision,double precision,numeric,numeric,double precision,double precision,double precision,numeric,numeric,double precision,numeric,numeric,numeric,numeric
6,dead_tree,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision
7,defect_ht_ft,double precision,numeric,double precision,double precision,numeric,numeric,double precision,double precision,double precision,numeric,numeric,double precision,numeric,numeric,numeric,numeric
8,defect_id,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text
9,disease_id,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text


In [20]:
# Conteo de presencia por fila
presence_cols = list(schemas.keys())
matrix["presence_count"] = matrix[presence_cols].apply(
    lambda r: sum(v != "" for v in r), axis=1
)

# Ordenar: primero las más comunes, luego las raras
matrix_sorted = (
    matrix
    .sort_values(
        by=["presence_count", "canonical_column"],
        ascending=[False, True]
    )
    .drop(columns="presence_count")
    .reset_index(drop=True)
)

matrix_sorted


,canonical_column,inventory_cr_2021,inventory_cr_2022,inventory_cr_2024,inventory_cr_2025,inventory_gt_2021,inventory_gt_2022,inventory_gt_2024,inventory_gt_2025,inventory_mx_2021,inventory_mx_2022,inventory_mx_2024,inventory_mx_2025,inventory_us_2021,inventory_us_2022,inventory_us_2024,inventory_us_2025
0,alive_tree,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision
1,contractcode,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text
2,coppiced_id,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text
3,cruisedate,date,date,timestamp without time zone,timestamp without time zone,date,date,timestamp without time zone,timestamp without time zone,date,date,timestamp without time zone,timestamp without time zone,date,date,timestamp without time zone,timestamp without time zone
4,dbh_in,double precision,numeric,double precision,double precision,numeric,numeric,double precision,double precision,double precision,numeric,numeric,double precision,numeric,numeric,numeric,numeric
5,dead_tree,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision,double precision
6,defect_ht_ft,double precision,numeric,double precision,double precision,numeric,numeric,double precision,double precision,double precision,numeric,numeric,double precision,numeric,numeric,numeric,numeric
7,defect_id,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text
8,disease_id,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text,text
9,doyle_bf,double precision,numeric,double precision,double precision,numeric,numeric,double precision,double precision,double precision,numeric,numeric,double precision,numeric,numeric,numeric,numeric


In [21]:
dfs = []

for _, row in inventories.iterrows():
    schema = row["schema"]
    table = row["table_name"]
    region = row["region"]
    year   = row["year"]

    print(f"📥 Loading {schema}.{table}")

    df = pd.read_sql(
        f'SELECT * FROM {schema}."{table}"',
        engine
    )

    # metadata útil
    df["source_table"] = table
    df["region"] = region
    df["inventory_year"] = year

    dfs.append(df)


📥 Loading public.inventory_cr_2021
📥 Loading public.inventory_cr_2022
📥 Loading public.inventory_cr_2024
📥 Loading public.inventory_cr_2025
📥 Loading public.inventory_gt_2021
📥 Loading public.inventory_gt_2022
📥 Loading public.inventory_gt_2024
📥 Loading public.inventory_gt_2025
📥 Loading public.inventory_mx_2021
📥 Loading public.inventory_mx_2022
📥 Loading public.inventory_mx_2024
📥 Loading public.inventory_mx_2025
📥 Loading public.inventory_us_2021
📥 Loading public.inventory_us_2022
📥 Loading public.inventory_us_2024
📥 Loading public.inventory_us_2025


In [22]:
df_unified = pd.concat(
    dfs,
    axis=0,
    ignore_index=True,
    sort=False   # mantiene todas las columnas, aunque falten en algunos
)

df_unified.shape


C:\Users\HeyCe\AppData\Local\Temp\ipykernel_15880\2145969431.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_unified = pd.concat(


(55196, 28)

##Primera corrección en batch: 1. Homogeneizar los plot_coordinates

In [23]:
def canonical_plot_coordinate(series: pd.Series):
    non_null = series.dropna()
    if non_null.empty:
        return None
    modes = non_null.mode()
    if not modes.empty:
        return modes.iloc[0]
    return non_null.iloc[0]


# Guardamos original para auditoría (opcional pero recomendado)
df_unified["plot_coordinate_original"] = df_unified["plot_coordinate"]

# Aplicamos homogenización
df_unified["plot_coordinate"] = (
    df_unified
    .groupby(["contractcode", "plot"])["plot_coordinate"]
    .transform(canonical_plot_coordinate)
)


In [24]:
after_check = (
    df_unified
    .groupby(["contractcode", "plot"])["plot_coordinate"]
    .nunique(dropna=True)
    .reset_index(name="distinct_plot_coordinates")
)


In [25]:
# comparison = (
#     before_check
#     .merge(
#         after_check,
#         on=["contractcode", "plot"],
#         suffixes=("_before", "_after")
#     )
# )
#
# comparison.query(
#     "distinct_plot_coordinates_before > 1"
# )[
#     [
#         "contractcode",
#         "plot",
#         "distinct_plot_coordinates_before",
#         "distinct_plot_coordinates_after"
#     ]
# ].sort_values(
#     "distinct_plot_coordinates_before",
#     ascending=False
# )


In [26]:
# remaining_conflicts = comparison.query(
#     "distinct_plot_coordinates_after > 1"
# )
#
# remaining_conflicts.shape


In [27]:
cols_to_drop = [
    "plot_coordinate_canonical",
    "plot_coordinate_original",
    "contract_plot_key",
]

df_unified = df_unified.drop(
    columns=[c for c in cols_to_drop if c in df_unified.columns]
)


In [28]:
##Corrección 2 checar y homogeneizar plots incompletos

In [29]:
plot_counts = (
    df_unified
    .groupby(["contractcode", "plot"])
    .size()
    .reset_index(name="trees_in_plot")
)

plot_stats = (
    plot_counts
    .groupby("contractcode")["trees_in_plot"]
    .agg(
        avg_trees_per_plot="mean",
        median_trees_per_plot="median",
        max_trees_per_plot="max",
        plots_in_contract="count"
    )
    .reset_index()
)


In [30]:
diagnosis = plot_counts.merge(
    plot_stats,
    on="contractcode",
    how="left"
)

diagnosis["flag_suspicious"] = (
    (diagnosis["trees_in_plot"] == 1) |
    (diagnosis["trees_in_plot"] < diagnosis["avg_trees_per_plot"] * 0.25)
)

diagnosis_suspicious = diagnosis.query("flag_suspicious")


In [31]:
summary = {
    "total_plots": diagnosis.shape[0],
    "suspicious_plots": diagnosis_suspicious.shape[0],
    "percent_suspicious": round(
        diagnosis_suspicious.shape[0] / diagnosis.shape[0] * 100, 2
    )
}

summary


{'total_plots': 3777, 'suspicious_plots': 93, 'percent_suspicious': 2.46}

In [32]:
df_unified.dtypes


contractcode          object
farmername            object
cruisedate            object
id                    object
id_error              object
stand                float64
plot                  object
plot_coordinate       object
tree_number          float64
defect_ht_ft         float64
dbh_in               float64
tht_ft               float64
merch_ht_ft          float64
short_note            object
Species_id            object
Defect_id             object
Pests_id              object
Coppiced_id           object
Permanent Plot_id     object
Disease_id            object
doyle_bf             float64
dead_tree            float64
alive_tree           float64
units_corrected       object
cruise_date           object
source_table          object
region                object
inventory_year         int64
dtype: object

## Vamos a guardar df_unified en staging y luego lo seguimos puliendo

In [33]:
df_unified.columns = (
    df_unified.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df_unified["cruisedate"] = pd.to_datetime(
    df_unified["cruisedate"],
    errors="coerce"
)

df_unified["cruise_date"] = pd.to_datetime(
    df_unified["cruise_date"],
    errors="coerce"
)


In [34]:
df_unified["units_corrected"] = (
    df_unified["units_corrected"]
    .astype(str)
    .str.lower()
    .map({"true": True, "false": False})
)


In [35]:
df_unified["stand"] = df_unified["stand"].astype("Int64").astype(str)


In [40]:

# 1) Vaciar staging
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE inventory.stg_inventory_unified_v1"))

# 2) Cargar df_unified
df_unified.to_sql(
    "stg_inventory_unified_v1",
    engine,
    schema="inventory",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000
)


ProgrammingError: (psycopg2.errors.UndefinedTable) relation "inventory.stg_inventory_unified_v1" does not exist

[SQL: TRUNCATE TABLE inventory.stg_inventory_unified_v1]
(Background on this error at: https://sqlalche.me/e/20/f405)